# 01 — Data Collection

Build the training dataset from `krishnareddy/icddxdescmap`.

## Inputs
- HuggingFace dataset: `krishnareddy/icddxdescmap` (~71k phrase→code pairs)

## Outputs
- `data/processed/train.jsonl`, `val.jsonl`, `test.jsonl` (chat-formatted)
- `data/processed/icd10_codebook.parquet` (code → descriptions lookup)

## Runtime
~5–10 min on CPU. No GPU needed.

## ⚠️ About this dataset

Labels come from a rule-based NLP extractor, not human annotation — expect
5–15% noisy labels. We embrace this as a teaching moment: real-world
healthcare data is never clean. The agent in Notebook 4 (with tool validation)
helps catch some of the noise downstream.


## 0. Bootstrap

Assumes you've run `00_setup.ipynb` this session. If not, run it first.
This cell just re-establishes cwd and sys.path in case the kernel restarted.


In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/icd10-slm')
assert PROJECT.exists(), "Run 00_setup.ipynb first"

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# Re-mount + re-auth if needed (safe to run twice)
if not os.getenv('HF_TOKEN'):
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=False)
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

from src import config
print(f"✓ ready  |  target codes: {config.TARGET_NUM_CODES}")


## 1. Load the raw dataset

The dataset provides its own train/val/test splits. We pull all three and
concatenate — we'll do our own stratified split over the combined pool.


In [ ]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd

splits = load_dataset("krishnareddy/icddxdescmap")
print("Built-in splits:", {k: len(v) for k, v in splits.items()})

raw = concatenate_datasets([splits[s] for s in splits.keys()])
print(f"\nTotal rows: {len(raw):,}")
print(f"Columns:    {raw.column_names}")

pd.DataFrame(raw[:5])


## 2. Normalize columns

Defensive rename that handles the dataset's lowercase columns (`docdesc`,
`dxcode`, …) or the README's camelCase (`AnnotationString`, `DXCode`, …).


In [ ]:
df = raw.to_pandas()
rename_map = {}
for col in df.columns:
    low = col.lower()
    if low in ('annotationstring', 'docdesc'):    rename_map[col] = 'clinical_text'
    elif low in ('dxcode', 'code'):                rename_map[col] = 'code'
    elif low in ('shortdesc', 'short_desc'):       rename_map[col] = 'short_desc'
    elif low in ('longdesc', 'long_desc'):         rename_map[col] = 'long_desc'

df = df.rename(columns=rename_map)
assert {'clinical_text', 'code', 'short_desc', 'long_desc'}.issubset(df.columns)
print(f"✓ Columns: {df.columns.tolist()}")
print(f"  Shape:   {df.shape}")


## 3. Clean & deduplicate

In [ ]:
before = len(df)

df['clinical_text'] = df['clinical_text'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
df['code']          = df['code'].astype(str).str.strip().str.upper()
df['short_desc']    = df['short_desc'].astype(str).str.strip()
df['long_desc']     = df['long_desc'].astype(str).str.strip()

df = df[df['clinical_text'].str.len() > 0]
df = df[df['code'].str.len() > 0]
df = df.drop_duplicates(subset=['clinical_text', 'code']).reset_index(drop=True)

print(f"After cleaning:  {len(df):,}  ({before - len(df):,} dropped)")
print(f"Unique codes:    {df['code'].nunique():,}")


## 4. Light quality filters

Cheap heuristics to drop obvious junk. Conservative — we want to keep ~95% of rows.


In [ ]:
import re

ICD10_RE = re.compile(r"^[A-TV-Z][0-9][A-Z0-9](\.[A-Z0-9]{1,4})?$")

drops = {
    'truncated':      df['clinical_text'].str.rstrip().str.endswith(('"', "'", ',', '-')),
    'too short':      df['clinical_text'].str.len() < 3,
    'digits only':    df['clinical_text'].str.replace(' ', '').str.isdigit(),
    'invalid code':   ~df['code'].str.match(ICD10_RE),
}
for name, mask in drops.items():
    print(f"  {name:20s}: {mask.sum():>5,} rows")

total_drop = drops['truncated'] | drops['too short'] | drops['digits only'] | drops['invalid code']
df = df[~total_drop].reset_index(drop=True)
print(f"\nKept: {len(df):,}  ({total_drop.sum():,} dropped = {total_drop.sum()/(len(df)+total_drop.sum())*100:.1f}%)")


## 5. EDA — the long-tail problem 🎯

**The money shot for teaching.** The plot below will look Zipf-like: a handful
of codes dominate, thousands have 1-5 examples. This is why Notebook 4 needs
RAG — fine-tuning alone can't cover the tail.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

code_counts = df['code'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(code_counts) + 1), code_counts.values, linewidth=2)
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_title('ICD-10 code rank vs. frequency', fontsize=12)
axes[0].set_xlabel('Rank (1 = most common)')
axes[0].set_ylabel('Examples')
axes[0].grid(True, which='both', alpha=0.3)

axes[1].hist(code_counts.values,
             bins=np.logspace(0, np.log10(code_counts.max()), 40),
             edgecolor='black')
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_title('Examples per code', fontsize=12)
axes[1].set_xlabel('Examples per code'); axes[1].set_ylabel('Number of codes')
axes[1].axvline(config.MIN_EXAMPLES_PER_CODE, color='red', linestyle='--',
                label=f'threshold = {config.MIN_EXAMPLES_PER_CODE}')
axes[1].legend()
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

total = len(df)
rare = (code_counts < config.MIN_EXAMPLES_PER_CODE).sum()
print("📊 THE LONG TAIL IN NUMBERS")
print("=" * 55)
print(f"Unique codes:                      {len(code_counts):,}")
print(f"Top 20 codes cover:                {code_counts.head(20).sum()/total*100:.1f}% of data")
print(f"Top 100 codes cover:               {code_counts.head(100).sum()/total*100:.1f}% of data")
print(f"Codes with < {config.MIN_EXAMPLES_PER_CODE} examples:       "
      f"{rare:,}  ({rare/len(code_counts)*100:.1f}% of codes)")
print()
print("☝ This is why Notebook 4 needs RAG.")


## 6. Filter to ~5000 mid-frequency codes

Rules (from `src/config.py`):
- Drop the top 20 trivially common codes (e.g. R10.9 "Unspec. abdominal pain")
- Drop codes with fewer than 5 examples
- Keep the next 5000 most common of what remains


In [ ]:
top_exclude = set(code_counts.head(config.TOP_K_CODES_TO_EXCLUDE).index)
print(f"Excluding top {config.TOP_K_CODES_TO_EXCLUDE} codes, samples:")
for c in list(top_exclude)[:5]:
    desc = df[df['code'] == c]['short_desc'].iloc[0]
    print(f"  {c:8s}  {code_counts[c]:>6,} ex   {desc[:60]}")

eligible = code_counts[
    (code_counts >= config.MIN_EXAMPLES_PER_CODE)
    & (~code_counts.index.isin(top_exclude))
]
kept_codes = set(eligible.head(config.TARGET_NUM_CODES).index)

df_filt = df[df['code'].isin(kept_codes)].reset_index(drop=True)
print(f"\nSelected codes:       {len(kept_codes):,}")
print(f"Examples kept:        {len(df_filt):,}  ({len(df_filt)/len(df)*100:.1f}%)")


## 7. Build the codebook

In [ ]:
codebook = (
    df_filt[['code', 'short_desc', 'long_desc']]
    .drop_duplicates('code').sort_values('code').reset_index(drop=True)
)
codebook['category'] = codebook['code'].str[:3]

print(f"Codebook: {codebook.shape}  |  Categories: {codebook['category'].nunique()}")
print()
print("Random sample:")
print(codebook.sample(5, random_state=42).to_string(index=False))


## 8. Format with Qwen3 chat template

Each example becomes a 3-turn chat: system prompt (from config) → clinical
text → ICD-10 code. We also measure token lengths to confirm
`MAX_SEQ_LENGTH=512` is safe.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL, trust_remote_code=True)

def to_chat(row):
    messages = [
        {"role": "system",    "content": config.SYSTEM_PROMPT},
        {"role": "user",      "content": row['clinical_text']},
        {"role": "assistant", "content": row['code']},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

df_filt['text']     = df_filt.apply(to_chat, axis=1)
df_filt['n_tokens'] = df_filt['text'].apply(lambda t: len(tokenizer.encode(t)))

print(f"Tokens — p50: {df_filt['n_tokens'].median():.0f} | "
      f"p95: {df_filt['n_tokens'].quantile(0.95):.0f} | "
      f"max: {df_filt['n_tokens'].max()}")
over = (df_filt['n_tokens'] > config.MAX_SEQ_LENGTH).sum()
print(f"Examples over MAX_SEQ_LENGTH ({config.MAX_SEQ_LENGTH}): {over}")
print()
print("--- sample formatted example ---")
print(df_filt['text'].iloc[0])


## 9. Stratified train/val/test split

Stratify by 3-char category so each split has similar clinical-specialty mix.
80 / 10 / 10 split.


In [ ]:
from sklearn.model_selection import train_test_split

df_filt['strat'] = df_filt['code'].str[:3]
rare = df_filt['strat'].value_counts()
rare_cats = rare[rare < 3].index.tolist()
df_filt.loc[df_filt['strat'].isin(rare_cats), 'strat'] = 'RARE'

train_val, test = train_test_split(df_filt, test_size=0.10,
                                   stratify=df_filt['strat'], random_state=42)
train, val = train_test_split(train_val, test_size=0.10/0.90,
                              stratify=train_val['strat'], random_state=42)

for name, part in [('train', train), ('val', val), ('test', test)]:
    print(f"{name:6s}: {len(part):>6,}  ({len(part)/len(df_filt)*100:.1f}%)")

unseen = set(test['code'].str[:3]) - set(train['code'].str[:3])
print(f"\nCategories in test unseen in train: {len(unseen)} {'⚠' if unseen else '✓'}")


## 10. Save artifacts

In [ ]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

save_cols = ['clinical_text', 'code', 'text']
for name, part, path in [
    ('train', train, config.TRAIN_PATH),
    ('val',   val,   config.VAL_PATH),
    ('test',  test,  config.TEST_PATH),
]:
    part[save_cols].to_json(path, orient='records', lines=True, force_ascii=False)
    print(f"✓ {name:6s}  {len(part):>6,} rows   →  {path.name}  "
          f"({path.stat().st_size/1024:,.0f} KB)")

codebook.to_parquet(config.CODEBOOK_PATH, index=False)
print(f"✓ codebook  {len(codebook):,} rows   →  {config.CODEBOOK_PATH.name}  "
      f"({config.CODEBOOK_PATH.stat().st_size/1024:,.0f} KB)")


## 11. Verify by reloading

In [ ]:
import json

for name, path, expected in [
    ('train', config.TRAIN_PATH, len(train)),
    ('val',   config.VAL_PATH,   len(val)),
    ('test',  config.TEST_PATH,  len(test)),
]:
    with open(path) as f:
        rows = [json.loads(line) for line in f]
    assert len(rows) == expected
    assert all(set(r) == {'clinical_text', 'code', 'text'} for r in rows)
    print(f"✓ {name}.jsonl: {len(rows):,} rows OK")

cb = pd.read_parquet(config.CODEBOOK_PATH)
assert set(cb.columns) == {'code', 'short_desc', 'long_desc', 'category'}
print(f"✓ codebook:    {len(cb):,} rows OK")


## ✅ Done

You have:
- `data/processed/train.jsonl`
- `data/processed/val.jsonl`
- `data/processed/test.jsonl`
- `data/processed/icd10_codebook.parquet`

## Commit your notebook changes (NOT the data — it's gitignored)

```python
!git add notebooks/01_data_collection.ipynb
!git commit -m "Run Notebook 1 — data collection complete"
!git push
```

## Next

Switch runtime to **L4 GPU** (`Runtime → Change runtime type → L4 GPU`),
then open `02_finetuning.ipynb`.
